# Notebook 23 — DualGraphEnergyNet: پیش‌بینی IFC از طریق مشتق دوم انرژی کاذب (Hessian)

## زمینه و انگیزه
در v1-v4 (notebook 15) و آزمایش‌های tuning (notebook 21)، ماتریس IFC **مستقیماً** توسط شبکه
پیش‌بینی می‌شد و قیدهای فیزیکی (تقارن، ASR) با جریمه‌های دستی (`λ_symmetry`, `λ_asr`) در
تابع loss اعمال می‌شدند — که هیچ‌وقت این قیود را به‌طور کامل و دقیق برقرار نکردند.

این notebook رویکرد را عوض می‌کند: به‌جای پیش‌بینی مستقیم IFC، شبکه فقط یک **عدد اسکالر**
(انرژی کاذب $U$) تولید می‌کند، و ماتریس IFC از طریق **مشتق دوم** این انرژی نسبت به مختصات
اتمی به دست می‌آید:

$$\Phi_{ij} = \frac{\partial^2 U}{\partial \mathbf{R}_i \partial \mathbf{R}_j}$$

این دقیقاً همان رویکردی است که در مدل‌های معتبر MLIP (مثل eSEN، NequIP، MACE) برای استخراج
ثابت‌های نیرو از یک تابع انرژی یادگرفته‌شده استفاده می‌شود.

## چرا این کار قیود فیزیکی را «رایگان» تضمین می‌کند
- **تقارن** `Φᵢⱼ = Φⱼᵢᵀ`: از قضیه‌ی Clairaut/Schwarz (تساوی مشتقات جزئی مرتبه‌دوم مختلط)
  به‌طور ریاضی نتیجه می‌شود — نیازی به جریمه‌ی دستی نیست.
- **قانون جمع آکوستیک (ASR)**: اگر $U$ فقط تابع فاصله‌های نسبی باشد (نه مختصات مطلق)،
  ASR معادل ریاضی invariance نسبت به انتقال کل سیستم است و خودکار برقرار می‌شود.
- این می‌تواند دقیقاً همان مشکلی را حل کند که در notebooks 16-18 کشف شد: پیش‌بینی
  factorized (مستقل برای هر اتم) هماهنگی سراسری را از بین می‌برد؛ اینجا یک تابع انرژی
  *واحد* همه‌ی اتم‌ها را به‌هم مرتبط نگه می‌دارد.

## ⚠️ تغییرات مهندسی لازم (که در گفتگوی قبلی مطرح شد)
1. **`BatchNorm1d` → `LayerNorm`**: BatchNorm با double-backward و محاسبه‌ی تک‌نمونه‌ای
   (batch=1 در حلقه‌ی هسین) ناسازگار است؛ همه‌جا با LayerNorm جایگزین شد.
2. **محاسبه‌ی دینامیک فاصله‌ها**: در `atom_graph`، فاصله‌های جفتی دیگر از پیش محاسبه و
   ثابت (`edge_attr` استاتیک) نیستند؛ داخل تابع انرژی، از تنسور خام `positions` (با
   `requires_grad=True`) و `torch.norm` محاسبه می‌شوند تا مسیر مشتق‌گیری کامل باشد.
3. **حلقه‌ی per-sample برای Hessian**: `torch.autograd.functional.hessian` روی کل یک
   batch به‌صورت یکجا باعث انفجار حافظه می‌شود؛ به‌جایش روی تک‌تک نمونه‌ها ایتریت می‌کنیم.
4. **آموزش دو-GPU دستی (T4 x2)**: چون `nn.DataParallel` با حلقه‌ی سفارشی per-sample
   Hessian به‌خوبی جور نیست، یک طرح ساده‌ی «data-parallel دستی» پیاده می‌شود: دو کپی از
   مدل (یکی روی هر GPU)، هر batch به دو نیم تقسیم می‌شود، هر نیمه روی یک GPU پردازش
   می‌شود، گرادیان‌ها در پایان جمع/میانگین‌گیری و فقط روی یک کپی apply می‌شوند، سپس وزن‌ها
   به کپی دوم کپی می‌شوند تا برای batch بعدی sync بمانند.

## ⚠️ هشدار هزینه‌ی محاسباتی
`hessian()` برای هر نمونه معادل چندین بار backward کامل شبکه است (به تعداد ۳n مختصات —
برای ۸ اتم یعنی ۲۴ بار، برای ۱۲ اتم یعنی ۳۶ بار). انتظار داشته باشید هر epoch **به‌طور
قابل‌توجهی کندتر** از v4 باشد. `EPOCHS`، `PATIENCE`، و `BATCH_SIZE` در این notebook محافظه‌کارانه
تنظیم شده‌اند؛ در صورت نیاز آن‌ها را متناسب با زمان واقعی مشاهده‌شده تنظیم کنید.


In [ ]:
!pip install -q phonopy torch_geometric
print("Installed: phonopy, torch_geometric")

In [ ]:
import os
import json
import yaml
import copy
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import Set2Set, global_mean_pool, global_max_pool
from torch_geometric.utils import softmax

from scipy.spatial.distance import cdist
from tqdm.notebook import tqdm

from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS

N_GPUS = torch.cuda.device_count()
print(f"Visible GPUs: {N_GPUS}")
for i in range(N_GPUS):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")

USE_DUAL_GPU = N_GPUS >= 2
DEVICE_0 = torch.device('cuda:0' if N_GPUS >= 1 else 'cpu')
DEVICE_1 = torch.device('cuda:1' if N_GPUS >= 2 else DEVICE_0)
print(f"\nDual-GPU manual data-parallel: {'ENABLED' if USE_DUAL_GPU else 'DISABLED (falling back to single device)'}")

## 1. Data paths (same as notebook 15/21)

In [ ]:
FC_DIR       = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
POSCAR_DIR   = '/kaggle/input/datasets/metisa81/pgcnndata/all_poscar/all_poscar'
BANDS_DIR    = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'
ELASTIC_FILE = '/kaggle/input/datasets/metisa81/features-dataset/mechanical_data_fixed.json'
FEATURE_CSV  = '/kaggle/input/datasets/metisa81/feature-dataset-split/5_geometry_radius_volume.csv'

for name, path in [('FC_DIR', FC_DIR), ('POSCAR_DIR', POSCAR_DIR),
                    ('BANDS_DIR', BANDS_DIR), ('ELASTIC_FILE', ELASTIC_FILE),
                    ('FEATURE_CSV', FEATURE_CSV)]:
    exists = 'OK' if os.path.exists(path) else 'MISSING'
    print(f"[{exists}]  {name}  ->  {path}")

## 2. Supercell auto-discovery + Phonopy loader (identical to notebook 15/21)

In [ ]:
def find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2_expected):
    """
    Find the correct diagonal supercell_matrix [[a,0,0],[0,b,0],[0,0,c]]
    by matching frequencies at a non-Gamma q-point.
    Returns: (a,b,c) or None if nothing matched.
    """
    n_images = n2_expected // len(unitcell.symbols)

    candidates = []
    for a in range(1, n_images + 1):
        if n_images % a != 0:
            continue
        rem = n_images // a
        for b in range(1, rem + 1):
            if rem % b != 0:
                continue
            c = rem // b
            candidates.append((a, b, c))

    best_dim, best_err = None, np.inf
    for (a, b, c) in candidates:
        try:
            phonon = Phonopy(unitcell, supercell_matrix=[[a,0,0],[0,b,0],[0,0,c]])
            if len(phonon.supercell.symbols) != n2_expected:
                continue
            phonon.force_constants = fc
            phonon.run_qpoints([q_test])
            freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
            if len(freqs) != len(real_freqs_test):
                continue
            err = np.max(np.abs(freqs - real_freqs_test))
            if err < best_err:
                best_err = err
                best_dim = (a, b, c)
        except Exception:
            continue

    return best_dim, best_err


def load_material_with_phonopy(yaml_path, fc_path):
    """Load one material with Phonopy: discover correct dim + build a ready Phonopy object."""
    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    lattice = np.array(data['lattice'])
    symbols = [p['symbol'] for p in data['points']]
    frac_coords = np.array([p['coordinates'] for p in data['points']])
    masses = [p['mass'] for p in data['points']]
    n1 = len(symbols)

    fc = parse_FORCE_CONSTANTS(str(fc_path))
    n2 = fc.shape[1]
    if fc.shape[0] != n1:
        return None

    unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

    segment_len = data['segment_nqpoint'][0]
    q_idx = min(segment_len // 2, len(data['phonon']) - 1)
    q_test = data['phonon'][q_idx]['q-position']
    real_freqs_test = np.sort([b['frequency'] for b in data['phonon'][q_idx]['band']])

    dim, err = find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2)
    if dim is None or err > 0.01:
        return None

    phonon = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
    phonon.force_constants = fc

    return {
        'phonon': phonon,
        'unitcell': unitcell,
        'symbols': symbols,
        'frac_coords': frac_coords,
        'lattice': lattice,
        'masses': masses,
        'supercell_dim': dim,
        'dim_match_error': err,
        'force_constants': fc,
        'all_q_points': [p['q-position'] for p in data['phonon']],
        'all_real_freqs': np.array([sorted([b['frequency'] for b in p['band']]) for p in data['phonon']]),
    }

print("Supercell discovery + Phonopy loader ready")

## 3. Build full dataset (358 materials, same filters as notebook 15/21)

In [ ]:
POSCAR_DIR_PATH = Path(POSCAR_DIR)
FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}
poscar_files = {f.stem: f for f in POSCAR_DIR_PATH.glob('*.psc')}

common = sorted(set(fc_files) & set(band_yaml_files) & set(poscar_files))
print(f"Materials with .fc + .yaml + .psc: {len(common)}")

with open(ELASTIC_FILE) as f:
    elastic_json = json.load(f)
ELASTIC_KEYS = ['C11','C12','C13','C33','C44','C66',
                'B_Hill','G_Hill','E_Hill','Poisson_Hill',
                'Pugh_ratio','Debye_temperature']
elastic_data = {}
for entry in elastic_json:
    mat = entry.get('material', '')
    vals = [float(entry.get(k, 0) or 0) for k in ELASTIC_KEYS]
    elastic_data[mat] = np.array(vals, dtype=np.float32)

In [ ]:
MAX_ATOMS_FOR_IFC = 12

raw_samples = []
failed_phonopy = []
failed_other = []

for formula in tqdm(common, desc="Loading materials with Phonopy"):
    try:
        with open(band_yaml_files[formula]) as f:
            quick_check = yaml.safe_load(f)
        if quick_check['natom'] > MAX_ATOMS_FOR_IFC:
            continue

        result = load_material_with_phonopy(band_yaml_files[formula], fc_files[formula])
        if result is None:
            failed_phonopy.append(formula)
            continue

        positions_cart = result['frac_coords'] @ result['lattice']

        raw_samples.append({
            'formula':           formula,
            'n_atoms':           len(result['symbols']),
            'lattice':           result['lattice'].astype(np.float32),
            'positions':         positions_cart.astype(np.float32),
            'atom_elements':     result['symbols'],
            'masses':            np.array(result['masses'], dtype=np.float32),
            'force_constants':   result['force_constants'].astype(np.float32),
            'supercell_dim':     result['supercell_dim'],
            'elastic_constants': elastic_data.get(formula, np.zeros(len(ELASTIC_KEYS), np.float32)),
            'y_phonon':          result['all_real_freqs'].astype(np.float32),
            'phonon_obj':        result['phonon'],
        })
    except Exception as e:
        failed_other.append((formula, str(e)))

print(f"\nOK: {len(raw_samples)} | failed_phonopy: {len(failed_phonopy)} | failed_other: {len(failed_other)}")

## 4. Train / Val / Test split

از همان `RANDOM_SEED=42` نسخه v4/v21 استفاده می‌شود تا مقایسه‌ی MAE با baseline (0.909،
یا 0.8903 بعد از tuning) معتبر بماند.

In [ ]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
idx = rng.permutation(len(raw_samples))

n_total = len(raw_samples)
n_tr = int(0.8 * n_total)
n_va = int(0.1 * n_total)

train_idx = idx[:n_tr].tolist()
val_idx   = idx[n_tr:n_tr+n_va].tolist()
test_idx  = idx[n_tr+n_va:].tolist()

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

## 5. Atomic feature table + static graph skeletons

⚠️ **تفاوت کلیدی با notebook 15/21:** `atom_graph` اینجا فقط `edge_index` (اتصال‌ها، ثابت
و بر اساس فاصله‌ی اولیه‌ی cutoff) و `x` (فیچرهای اتمی ثابت) را نگه می‌دارد. `edge_attr`
(فاصله‌های جفتی) دیگر اینجا ذخیره نمی‌شود — این‌ها هر بار داخل تابع انرژی، از روی تنسور
`positions` قابل‌مشتق‌گیری، با `torch.norm` محاسبه می‌شوند.

In [ ]:
CUTOFF = 8.0

df_raw = pd.read_csv(FEATURE_CSV)
symbol_col = next((c for c in df_raw.columns if c.lower() in ['symbol', 'element', 'atom']), df_raw.columns[0])
feature_cols = [c for c in df_raw.columns if c not in [symbol_col, 'atomic_number']]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_raw[c])]
df_features = df_raw[feature_cols].copy()
df_features.index = df_raw[symbol_col]
N_ATOM_FEATURES = len(feature_cols)
print(f"Atomic features: {N_ATOM_FEATURES} -> {feature_cols}")


def structure_to_bond_graph(positions, n_atoms_sample):
    """Static bond graph (unchanged from v4): fixed geometric context, not part of the
    differentiable energy path. Provides auxiliary structural conditioning only."""
    n = len(positions)
    distances = cdist(positions, positions)
    bonds = [(i, j) for i in range(n) for j in range(i+1, n) if distances[i, j] <= CUTOFF]

    if len(bonds) == 0:
        x = torch.zeros((1, 6), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
        u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u)

    node_features = []
    for i, j in bonds:
        d = distances[i, j]
        coord_i = np.sum(distances[i] <= CUTOFF) - 1
        coord_j = np.sum(distances[j] <= CUTOFF) - 1
        node_features.append([d, coord_i, coord_j, (coord_i + coord_j) / 2, 0.0, 0.0])
    x = torch.tensor(node_features, dtype=torch.float)

    edge_index, edge_attr = [], []
    for idx1, (i1, j1) in enumerate(bonds):
        for idx2, (i2, j2) in enumerate(bonds[idx1+1:]):
            idx2 += idx1 + 1
            shared = set([i1, j1]) & set([i2, j2])
            if len(shared) == 1:
                s = shared.pop()
                a = positions[i1 if i1 != s else j1] - positions[s]
                b = positions[i2 if i2 != s else j2] - positions[s]
                na, nb = np.linalg.norm(a), np.linalg.norm(b)
                if na > 0 and nb > 0:
                    cos_angle = np.clip(np.dot(a, b) / (na * nb), -1.0, 1.0)
                    angle = np.arccos(cos_angle)
                    edge_index.extend([[idx1, idx2], [idx2, idx1]])
                    edge_attr.extend([[angle], [angle]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous() if edge_index else torch.zeros((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float) if edge_attr else torch.zeros((0, 1), dtype=torch.float)
    u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u)


def atom_graph_skeleton(atom_elements, positions):
    """Static atom-graph skeleton: x (atomic features) + edge_index (fixed connectivity
    from initial cutoff distances). NO edge_attr here -- distances are recomputed
    dynamically inside the energy function from the differentiable positions tensor."""
    node_features = []
    for elem in atom_elements:
        if elem in df_features.index:
            feats = df_features.loc[elem].values.astype(np.float32)
        else:
            feats = np.zeros(N_ATOM_FEATURES, dtype=np.float32)
        node_features.append(feats)
    x = torch.tensor(np.array(node_features), dtype=torch.float)

    distances = cdist(positions, positions)
    edge_index = []
    n = len(positions)
    for i in range(n):
        for j in range(n):
            if i != j and distances[i, j] <= CUTOFF:
                edge_index.append([i, j])

    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    return x, edge_index


print("Graph skeleton functions ready")

In [ ]:
bond_graphs = []
atom_x_list, atom_edge_index_list = [], []
positions_list, ifc_targets = [], []

for s in tqdm(raw_samples, desc="Building graph skeletons"):
    bond_graphs.append(structure_to_bond_graph(s['positions'], s['n_atoms']))
    x, edge_index = atom_graph_skeleton(s['atom_elements'], s['positions'])
    atom_x_list.append(x)
    atom_edge_index_list.append(edge_index)
    positions_list.append(torch.tensor(s['positions'], dtype=torch.float32))
    ifc_targets.append(s['force_constants'])

MAX_ATOMS = max(s['n_atoms'] for s in raw_samples)
print(f"Skeletons built: {len(bond_graphs)} | max_atoms = {MAX_ATOMS}")

## 6. معماری: DualGraphEnergyNet

- `CustomMessagePassing` بدون تغییر است، به‌جز اینکه لایه‌های نرمال‌سازی داخلی به‌طور
  کامل حذف شده‌اند (خود `LayerNorm` در `_encode_graph` باقی می‌ماند که مشکلی با
  double-backward ندارد).
- **`BatchNorm1d` → `LayerNorm`** در `bond_embedding`/`atom_embedding` (چون BatchNorm با
  batch=1 در حالت train خطا می‌دهد و در eval از آمار running استفاده می‌کند که برای
  مشتق‌گیری پایدار مناسب نیست).
- سر خروجی از `ifc_head` (که یک ماتریس بزرگ تخت می‌داد) به `energy_head` تغییر کرده:
  فقط **یک عدد اسکالر** (انرژی کاذب) تولید می‌کند.

In [ ]:
class CustomMessagePassing(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.attention_mlp = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.LeakyReLU(0.2)
        )
        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        source_nodes = edge_index[0]
        target_nodes = edge_index[1]
        neighbor_features = x[source_nodes]
        target_features = x[target_nodes]

        attention_input = torch.cat([target_features, neighbor_features, edge_attr], dim=1)
        attention_scores = self.attention_mlp(attention_input)
        attention_weights = softmax(attention_scores, target_nodes, num_nodes=x.size(0))

        messages = neighbor_features * edge_attr
        weighted_messages = messages * attention_weights

        num_nodes = x.size(0)
        # NOTE: use out-of-place scatter-add (index_add on a freshly allocated zero
        # tensor) rather than an in-place op on a tensor that requires grad, so the
        # computational graph stays valid for double-backward (Hessian).
        aggregated = torch.zeros(num_nodes, self.hidden_dim, device=x.device, dtype=x.dtype)
        aggregated = aggregated.index_add(0, target_nodes, weighted_messages)

        return self.message_mlp(aggregated)


class DualGraphEnergyNet(nn.Module):
    """Refactored from DualGraphIFCNet: outputs a single scalar pseudo-energy U instead
    of the full IFC matrix. The IFC is obtained externally as the Hessian of U with
    respect to atomic positions (see energy_forward / compute_predicted_ifc below)."""

    def __init__(self, n_bond_features=6, n_atom_features=7, edge_dim=1,
                 hidden_dim=128, n_bond_layers=5, n_atom_layers=2, dropout=0.1,
                 set2set_steps=1):
        super().__init__()
        hidden = hidden_dim

        # BatchNorm1d -> LayerNorm: BatchNorm breaks with batch_size=1 (required for the
        # per-sample Hessian loop) and behaves inconsistently between train/eval mode
        # under double-backward. LayerNorm has no such batch-size dependency.
        self.bond_embedding = nn.Sequential(
            nn.Linear(n_bond_features, hidden), nn.LayerNorm(hidden), nn.SiLU(), nn.Dropout(dropout))
        self.bond_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.bond_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(n_bond_layers)])
        self.bond_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_bond_layers)])
        self.bond_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.atom_embedding = nn.Sequential(
            nn.Linear(n_atom_features, hidden), nn.LayerNorm(hidden), nn.SiLU(), nn.Dropout(dropout))
        self.atom_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.atom_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(n_atom_layers)])
        self.atom_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_atom_layers)])
        self.atom_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.bond_residual_weight = nn.Parameter(torch.tensor(0.3))
        self.atom_residual_weight = nn.Parameter(torch.tensor(0.3))

        self.set2set_pool = Set2Set(hidden, processing_steps=set2set_steps)
        self.mean_pool = global_mean_pool
        self.max_pool = global_max_pool
        self.global_mlp = nn.Sequential(nn.Linear(3, hidden // 4), nn.SiLU())

        combined_dim = 8 * hidden + hidden // 4

        # Energy head: single scalar output (pseudo-energy U). IFC is obtained
        # externally as the Hessian of this scalar w.r.t. atomic positions.
        self.energy_head = nn.Sequential(
            nn.Linear(combined_dim, 256), nn.LayerNorm(256), nn.SiLU(),
            nn.Linear(256, 64), nn.LayerNorm(64), nn.SiLU(),
            nn.Linear(64, 1)
        )

    def _encode_graph(self, x, edge_index, edge_attr, embedding, edge_embedding,
                       message_layers, layer_norms, attention, residual_weight):
        x = embedding(x)
        edge_feat = edge_embedding(edge_attr)
        for i, (msg, ln) in enumerate(zip(message_layers, layer_norms)):
            x_res = x
            x = msg(x, edge_index, edge_feat)
            x = ln(x)
            if i > 0:
                x = x + residual_weight * x_res
            x = F.silu(x)
        attn = attention(x)
        return x * attn, x

    def forward(self, bond_data, atom_data):
        bond_w, x_bond = self._encode_graph(
            bond_data.x, bond_data.edge_index, bond_data.edge_attr,
            self.bond_embedding, self.bond_edge_embedding,
            self.bond_message_layers, self.bond_layer_norms,
            self.bond_attention, self.bond_residual_weight)

        bond_set2set = self.set2set_pool(bond_w, bond_data.batch)
        bond_mean = self.mean_pool(x_bond, bond_data.batch)
        bond_max = self.max_pool(x_bond, bond_data.batch)

        atom_w, x_atom = self._encode_graph(
            atom_data.x, atom_data.edge_index, atom_data.edge_attr,
            self.atom_embedding, self.atom_edge_embedding,
            self.atom_message_layers, self.atom_layer_norms,
            self.atom_attention, self.atom_residual_weight)

        atom_set2set = self.set2set_pool(atom_w, atom_data.batch)
        atom_mean = self.mean_pool(x_atom, atom_data.batch)
        atom_max = self.max_pool(x_atom, atom_data.batch)

        global_features = self.global_mlp(bond_data.u)

        combined = torch.cat([
            bond_set2set, bond_mean, bond_max,
            atom_set2set, atom_mean, atom_max,
            global_features
        ], dim=1)

        energy = self.energy_head(combined)  # shape (1, 1) for a single sample
        return energy.view(())  # scalar tensor

print("DualGraphEnergyNet ready")

## 7. تابع انرژی و استخراج هسین (IFC)

`energy_forward` دقیقاً یک نمونه را می‌گیرد. فاصله‌های جفتی `atom_graph` داخل همین تابع،
از تنسور `positions` (که `requires_grad=True` دارد) با `torch.norm` محاسبه می‌شوند —
این یعنی مسیر محاسباتی PyTorch از موقعیت اتم‌ها تا انرژی نهایی کاملاً قابل ردیابی
(differentiable) است و `torch.autograd.functional.hessian` می‌تواند مشتق دوم را استخراج
کند.

In [ ]:
def energy_forward(positions, atom_x, atom_edge_index, bond_data, model, device):
    """Compute the scalar pseudo-energy for ONE sample.

    positions: (n_atoms, 3) tensor, requires_grad=True -- this is the tensor we will
               differentiate through (twice) to get the IFC via its Hessian.
    """
    src, dst = atom_edge_index[0], atom_edge_index[1]
    # Dynamic pairwise distances computed from the differentiable positions tensor.
    # This is the critical step that keeps the autograd graph connected from
    # positions -> distances -> message passing -> energy.
    diff = positions[dst] - positions[src]
    dyn_dist = torch.norm(diff, dim=1, keepdim=True)

    atom_data = Data(x=atom_x, edge_index=atom_edge_index, edge_attr=dyn_dist)
    atom_data.batch = torch.zeros(atom_x.size(0), dtype=torch.long, device=device)
    bond_data.batch = torch.zeros(bond_data.x.size(0), dtype=torch.long, device=device)

    energy = model(bond_data, atom_data)
    return energy


def compute_predicted_ifc(model, positions, atom_x, atom_edge_index, bond_data, device,
                           create_graph):
    """Compute the Hessian of the energy w.r.t. positions and reshape it into the
    (n_atoms, n_atoms, 3, 3) IFC block format used throughout the project.

    create_graph=True is required during training so the MSE loss on the Hessian can
    backpropagate a second time into the model's weights (double-backward). During
    evaluation (no further backward needed), create_graph=False saves memory.
    """
    n_atoms = positions.shape[0]

    def energy_fn(p):
        return energy_forward(p, atom_x, atom_edge_index, bond_data, model, device)

    # torch.autograd.functional.hessian on a (n_atoms, 3) input returns a tensor of
    # shape (n_atoms, 3, n_atoms, 3) -- the full second-derivative tensor.
    hess = torch.autograd.functional.hessian(
        energy_fn, positions, create_graph=create_graph, vectorize=False)

    # Reorder (n_atoms, 3, n_atoms, 3) -> (n_atoms, n_atoms, 3, 3) to match the
    # true_ifc target format used in notebooks 15/21.
    ifc_pred = hess.permute(0, 2, 1, 3)
    return ifc_pred

print("Energy function + Hessian extraction ready")

## 8. آموزش دو-GPU دستی (T4 x2)

از یک طرح ساده‌ی «data-parallel دستی» استفاده می‌شود (جایگزین سبک‌تر برای
`DistributedDataParallel`، مناسب همین حلقه‌ی سفارشی per-sample Hessian):

1. دو کپی از مدل با وزن‌های یکسان — یکی روی `cuda:0`، یکی روی `cuda:1`
2. هر mini-batch به دو نیم تقسیم می‌شود؛ هر نیمه (با حلقه‌ی per-sample) روی یک GPU
   پردازش می‌شود
3. بعد از `backward()` هر دو نیمه، گرادیان‌های کپی دوم به دستگاه اول منتقل و با کپی اول
   جمع/میانگین‌گیری می‌شوند
4. `optimizer.step()` فقط روی کپی اول اجرا می‌شود؛ سپس وزن‌های به‌روزشده به کپی دوم کپی
   می‌شوند تا برای batch بعدی همگام (sync) بمانند

اگر فقط یک GPU در دسترس باشد (`USE_DUAL_GPU=False`)، این حلقه به‌طور خودکار روی حالت
تک-GPU ساده fallback می‌کند.

In [ ]:
def move_sample_to_device(i, device):
    positions = positions_list[i].clone().detach().to(device).requires_grad_(True)
    atom_x = atom_x_list[i].to(device)
    atom_edge_index = atom_edge_index_list[i].to(device)
    bond_data = bond_graphs[i].clone().to(device)
    true_ifc = torch.tensor(ifc_targets[i], dtype=torch.float32, device=device)
    n = positions.shape[0]
    true_ifc_block = true_ifc[:n, :n, :, :]
    return positions, atom_x, atom_edge_index, bond_data, true_ifc_block


def process_sample_list(model, indices, device, is_train):
    """Loop over a list of sample indices on a single device/model replica.
    Returns the mean MSE loss (as a tensor still attached to the graph if is_train)."""
    losses = []
    for i in indices:
        positions, atom_x, atom_edge_index, bond_data, true_ifc_block = move_sample_to_device(i, device)

        ifc_pred = compute_predicted_ifc(
            model, positions, atom_x, atom_edge_index, bond_data, device,
            create_graph=is_train)

        loss = F.mse_loss(ifc_pred, true_ifc_block)
        losses.append(loss)

        # Free the large per-sample intermediate graph promptly to avoid memory leaks
        # across the loop (important since create_graph=True keeps a much larger graph
        # alive than a normal single backward).
        del positions, atom_x, atom_edge_index, bond_data, true_ifc_block, ifc_pred

    mean_loss = torch.stack(losses).mean()
    return mean_loss


def sync_replica(model_main, model_secondary):
    model_secondary.load_state_dict(model_main.state_dict())


def run_epoch_dual_gpu(model_0, model_1, optimizer, indices, batch_size, shuffle, rng, is_train):
    idx_arr = np.array(indices)
    if shuffle:
        idx_arr = rng.permutation(idx_arr)

    total_loss, n_batches = 0.0, 0
    model_0.train() if is_train else model_0.eval()
    model_1.train() if is_train else model_1.eval()

    for start in range(0, len(idx_arr), batch_size):
        batch = idx_arr[start:start+batch_size].tolist()
        if USE_DUAL_GPU and len(batch) > 1:
            mid = len(batch) // 2
            half_0, half_1 = batch[:mid] if mid > 0 else batch, batch[mid:] if mid > 0 else []
        else:
            half_0, half_1 = batch, []

        if is_train:
            optimizer.zero_grad()
            for p in model_1.parameters():
                if p.grad is not None:
                    p.grad = None

        with torch.set_grad_enabled(is_train):
            loss_0 = process_sample_list(model_0, half_0, DEVICE_0, is_train) if half_0 else None
            loss_1 = process_sample_list(model_1, half_1, DEVICE_1, is_train) if half_1 else None

        if is_train:
            if loss_0 is not None:
                loss_0.backward()
            if loss_1 is not None:
                loss_1.backward()

            if loss_1 is not None:
                # Merge replica-1's gradients into replica-0 (manual all-reduce of 2).
                n0, n1 = len(half_0), len(half_1)
                total_n = n0 + n1
                for p0, p1 in zip(model_0.parameters(), model_1.parameters()):
                    g1 = p1.grad.to(DEVICE_0) if p1.grad is not None else None
                    if p0.grad is None:
                        p0.grad = torch.zeros_like(p0)
                    # Weighted average by how many samples each replica processed.
                    p0.grad.mul_(n0 / total_n)
                    if g1 is not None:
                        p0.grad.add_(g1, alpha=n1 / total_n)

            torch.nn.utils.clip_grad_norm_(model_0.parameters(), 1.0)
            optimizer.step()

            if USE_DUAL_GPU:
                sync_replica(model_0, model_1)

        combined_loss = 0.0
        combined_n = 0
        if loss_0 is not None:
            combined_loss += loss_0.item() * len(half_0)
            combined_n += len(half_0)
        if loss_1 is not None:
            combined_loss += loss_1.item() * len(half_1)
            combined_n += len(half_1)

        total_loss += combined_loss / max(combined_n, 1)
        n_batches += 1

        if DEVICE_0.type == 'cuda':
            torch.cuda.empty_cache()

    return total_loss / max(n_batches, 1)

print("Dual-GPU manual data-parallel training loop ready")

## 9. اجرای آموزش

⚠️ با توجه به هزینه‌ی بالای Hessian، مقادیر `EPOCHS`/`PATIENCE`/`BATCH_SIZE` محافظه‌کارانه
انتخاب شده‌اند. بعد از دیدن زمان واقعی هر epoch روی T4x2، این مقادیر را متناسب تنظیم کنید.

In [ ]:
EPOCHS = 150
PATIENCE = 30
BATCH_SIZE = 8   # split into 4+4 across the two GPUs when USE_DUAL_GPU
LR = 3e-4
WEIGHT_DECAY = 1e-4
SEED = 42

torch.manual_seed(SEED)
model_0 = DualGraphEnergyNet(
    n_bond_features=6, n_atom_features=N_ATOM_FEATURES, edge_dim=1,
    hidden_dim=128, n_bond_layers=5, n_atom_layers=2, dropout=0.1, set2set_steps=1
).to(DEVICE_0)

model_1 = DualGraphEnergyNet(
    n_bond_features=6, n_atom_features=N_ATOM_FEATURES, edge_dim=1,
    hidden_dim=128, n_bond_layers=5, n_atom_layers=2, dropout=0.1, set2set_steps=1
).to(DEVICE_1) if USE_DUAL_GPU else model_0

if USE_DUAL_GPU:
    sync_replica(model_0, model_1)  # ensure identical initial weights

optimizer = torch.optim.AdamW(model_0.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=12)

rng = np.random.default_rng(SEED)
best_val_loss = float('inf')
best_state = None
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss = run_epoch_dual_gpu(model_0, model_1, optimizer, train_idx, BATCH_SIZE, True, rng, is_train=True)
    val_loss = run_epoch_dual_gpu(model_0, model_1, optimizer, val_idx, BATCH_SIZE, False, rng, is_train=False)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model_0.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    print(f"Epoch {epoch:4d} | Train MSE(IFC): {train_loss:.5f} | Val MSE(IFC): {val_loss:.5f} | Best: {best_val_loss:.5f}")

    if patience_counter >= PATIENCE:
        print(f"Early stop at epoch {epoch}")
        break

model_0.load_state_dict(best_state)
torch.save(best_state, '/kaggle/working/energy_hessian_model.pt')
print("\nTraining done. Best model saved.")

## 10. ارزیابی نهایی: MAE فرکانس واقعی (با موتور Phonopy)

همان پروتکل ارزیابی v4/v21: ماتریس IFC پیش‌بینی‌شده (اینجا از طریق Hessian) جایگزین بلوک
سلول مرجع در ماتریس force_constants واقعی می‌شود و فرکانس از طریق خود Phonopy محاسبه
می‌شود.

In [ ]:
def predict_frequencies_with_phonopy(ifc_pred_np, sample):
    phonon = sample['phonon_obj']
    n = sample['n_atoms']
    ifc_for_phonopy = ifc_pred_np[:n, :n, :, :]
    fc_copy = sample['force_constants'].copy()
    fc_copy[:n, :n, :, :] = ifc_for_phonopy
    phonon.force_constants = fc_copy
    phonon.run_qpoints([[0, 0, 0]])
    freqs = phonon.get_qpoints_dict()['frequencies']
    return np.array(freqs)


def evaluate_freq_mae_hessian(model, indices, device):
    model.eval()
    all_errors = []
    for i in tqdm(indices, desc="Evaluating (Hessian -> Phonopy)", leave=False):
        s = raw_samples[i]
        positions, atom_x, atom_edge_index, bond_data, _ = move_sample_to_device(i, device)
        ifc_pred = compute_predicted_ifc(
            model, positions, atom_x, atom_edge_index, bond_data, device,
            create_graph=False)
        ifc_pred_np = ifc_pred.detach().cpu().numpy()

        try:
            pred_freqs = predict_frequencies_with_phonopy(ifc_pred_np, s)
            true_peak = float(np.max(s['y_phonon']))
            pred_peak = float(np.max(np.abs(pred_freqs)))
            all_errors.append(abs(true_peak - pred_peak))
        except Exception:
            continue

        del positions, atom_x, atom_edge_index, bond_data, ifc_pred

    return float(np.mean(all_errors)) if all_errors else float('nan')


test_freq_mae = evaluate_freq_mae_hessian(model_0, test_idx, DEVICE_0)
print(f"\n=== Final comparison (test set) ===")
print(f"  v4 baseline (direct IFC prediction, no tuning) : 0.909 THz")
print(f"  v4 + Optuna tuning (notebook 21, best solo)    : 0.8903 THz")
print(f"  DualGraphEnergyNet (Hessian-based, this nb)    : {test_freq_mae:.4f} THz")

## جمع‌بندی

این notebook معماری IFC را از پیش‌بینی مستقیم به یک رویکرد **مبتنی بر انرژی کاذب** بازنویسی
کرد: شبکه فقط یک عدد اسکالر تولید می‌کند، و ماتریس IFC از طریق مشتق دوم (Hessian) آن نسبت
به مختصات اتمی استخراج می‌شود. این کار قیود تقارن و ASR را به‌طور ریاضی (نه با جریمه‌ی
دستی) تضمین می‌کند.

### تغییرات مهندسی کلیدی نسبت به v4/v21
- `BatchNorm1d` → `LayerNorm` (سازگاری با double-backward و batch=1)
- فاصله‌های اتمی دیگر استاتیک نیستند؛ داخل تابع انرژی از `positions` قابل‌مشتق‌گیری
  محاسبه می‌شوند
- حلقه‌ی per-sample برای Hessian (به‌جای batched، برای جلوگیری از انفجار حافظه)
- آموزش دو-GPU دستی (T4 x2) با همگام‌سازی گرادیان و وزن بین دو کپی مدل

### نکات برای تفسیر نتیجه
- این رویکرد از نظر روش‌شناسی با یافته‌ی notebooks 16-18 سازگار است (که نشان داد
  factorization مستقل بین اتم‌ها هماهنگی فیزیکی را از بین می‌برد) — یک تابع انرژی واحد
  طبیعتاً این هماهنگی را حفظ می‌کند.
- هزینه‌ی محاسباتی هر epoch به‌طور قابل‌توجهی بالاتر از v4 است (هر نمونه معادل چندین بار
  backward کامل شبکه)؛ `EPOCHS`/`BATCH_SIZE` را متناسب با زمان واقعی مشاهده‌شده تنظیم کنید.
- توصیه می‌شود قبل از commit کامل، یک pilot کوچک (مثلاً ۱۰-۲۰ ماده، چند epoch) اجرا شود
  تا صحت عددی محاسبه‌ی Hessian (و هماهنگی دو-GPU) قبل از آموزش کامل تأیید شود.

### ثبت در Obsidian (بعد از اجرا روی Kaggle)
- MAE فرکانس نهایی و مقایسه با v4 (0.909) و v21 (0.8903)
- زمان واقعی هر epoch (برای برنامه‌ریزی آزمایش‌های بعدی)
- در صورت بهبود معنادار: این می‌تواند بخش مهمی از Methods/Discussion مقاله شود
  (رفع کامل و ریاضی قیود فیزیکی، به‌جای جریمه‌ی دستی)
